In [22]:
import re
import pandas as pd
from pathlib import Path

In [23]:
run_dir = Path(r"../data\processed_data\run_2026-04-28_153116")

In [24]:
sum_vars = [
    'total_residential_units', 'total_occ_units',
    'occ_units_low_inc', 'occ_units_med_inc', 'occ_units_high_inc',
    'total_persons',
    'emp_retail', 'emp_srvc', 'emp_rec', 'emp_game', 'emp_other',
]
emp_cols = ['emp_retail', 'emp_srvc', 'emp_rec', 'emp_game', 'emp_other']

records = {}

for scenario_dir in sorted(d for d in run_dir.iterdir() if d.is_dir() and d.name != 'configs'):
    clean_name = re.sub(r'_\d{8}', '', scenario_dir.name)
    for year in [2035, 2050]:
        csv_path = scenario_dir / str(year) / 'SocioEcon_Summer.csv'
        if not csv_path.exists():
            continue
        df = pd.read_csv(csv_path)
        key = f"{clean_name}_{year}"
        records[key] = {v: int(df[v].sum()) for v in sum_vars}
        records[key]['total_jobs'] = int(df[emp_cols].sum().sum())

comparison = pd.DataFrame(records, index=sum_vars + ['total_jobs'])
comparison.index.name = 'variable'

# Diff each scenario's 2050 vs scenario 1's 2050 (first sorted scenario)
cols_2050 = [c for c in comparison.columns if c.endswith('_2050')]
if len(cols_2050) >= 2:
    baseline = cols_2050[0]
    for col in cols_2050[1:]:
        label = col[: -len('_2050')]
        comparison[f'diff_{label}_vs_alt1_2050'] = comparison[col] - comparison[baseline]

comparison.to_csv(run_dir / 'scenario_comparison.csv')
print(f"Saved: {run_dir / 'scenario_comparison.csv'}")
comparison

Saved: ..\data\processed_data\run_2026-04-28_153116\scenario_comparison.csv


,Alternative_1_2035,Alternative_1_2050,Alternative_2_2035,Alternative_2_2050,Alternative_3_2035,Alternative_3_2050,diff_Alternative_2_vs_alt1_2050,diff_Alternative_3_vs_alt1_2050
variable,,,,,,,,
total_residential_units,51062,54197,51469,55435,51062,54197,1238,0
total_occ_units,24687,26507,24997,27750,25019,27362,1243,855
occ_units_low_inc,10563,11441,10709,11650,10563,11441,209,0
occ_units_med_inc,4889,5142,4952,5607,5042,5556,465,414
occ_units_high_inc,9235,9924,9336,10493,9414,10365,569,441
total_persons,55588,57608,56281,60244,56340,59473,2636,1865
emp_retail,3778,3855,3778,3855,3778,3855,0,0
emp_srvc,7632,7777,7632,7777,7632,7777,0,0
emp_rec,2286,2318,2286,2318,2286,2318,0,0


In [25]:
parquet_path = run_dir / 'all_scenarios_parcels.parquet'
if not parquet_path.exists():
    print(f"No parquet found at {parquet_path}")
else:
    df = pd.read_parquet(parquet_path)
    df['SCENARIO'] = df['SCENARIO'].apply(lambda s: re.sub(r'_\d{8}', '', s))
    df['reason_clean'] = (
        df['FORECAST_REASON']
        .fillna('')
        .str.replace(' Infill', '', regex=False)
        .str.replace(' MF', '', regex=False)
        .str.replace(' SF', '', regex=False)
        .str.strip()
    )

    # ── Residential units by reason × scenario ────────────────────────────────
    reason_summary = (
        df.groupby(['SCENARIO', 'reason_clean'])['FORECASTED_RESIDENTIAL_UNITS']
        .sum()
        .unstack('SCENARIO')
        .fillna(0)
        .astype(int)
    )
    reason_summary.index.name = 'forecast_reason'
    reason_summary.to_csv(run_dir / 'units_by_reason.csv')
    print(f"Saved: {run_dir / 'units_by_reason.csv'}")

    # ── Residential + income units by reason × scenario (wide) ───────────────
    income_cols = {
        'residential':  'FORECASTED_RESIDENTIAL_UNITS',
        'low_inc':      'FORECASTED_RES_INCOME_LOW_UNITS',
        'med_inc':      'FORECASTED_RES_INCOME_MEDIUM_UNITS',
        'high_inc':     'FORECASTED_RES_INCOME_HIGH_UNITS',
    }
    parts = []
    for metric, col in income_cols.items():
        pivot = (
            df.groupby(['SCENARIO', 'reason_clean'])[col]
            .sum()
            .unstack('SCENARIO')
            .fillna(0)
            .astype(int)
        )
        pivot.columns = [f"{s}_{metric}" for s in pivot.columns]
        parts.append(pivot)

    income_summary = pd.concat(parts, axis=1).fillna(0).astype(int)
    # Interleave columns so each scenario's 4 metrics stay together
    scenarios = df['SCENARIO'].unique()
    ordered_cols = [f"{s}_{m}" for s in sorted(scenarios) for m in income_cols]
    income_summary = income_summary[[c for c in ordered_cols if c in income_summary.columns]]
    income_summary.index.name = 'forecast_reason'
    income_summary.to_csv(run_dir / 'units_by_reason_income.csv')
    print(f"Saved: {run_dir / 'units_by_reason_income.csv'}")
    income_summary

Saved: ..\data\processed_data\run_2026-04-28_153116\units_by_reason.csv
Saved: ..\data\processed_data\run_2026-04-28_153116\units_by_reason_income.csv
